<a href="https://colab.research.google.com/github/too-baa/e-commerce-customer-segmentation/blob/main/CaseStudy4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import datetime as dt
import pandas as pd

# 1. Set environment variables for Kaggle authentication
os.environ['KAGGLE_USERNAME'] = "KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "KAGGLE_KEY"

# 2. Install kaggle CLI if not already present
!pip install -q kaggle

# 3. Download and unzip the official Online Retail dataset
!kaggle datasets download -d vijayuv/onlineretail -p ./ --unzip

# 4. Read the CSV file into Pandas
# The downloaded file is named 'OnlineRetail.csv'
df = pd.read_csv("OnlineRetail.csv", encoding="ISO-8859-1")

print("Raw Dataset Shape:", df.shape)
print(df.head(3))

Dataset URL: https://www.kaggle.com/datasets/vijayuv/onlineretail
License(s): CC0-1.0
100% 7.20M/7.20M [00:00<00:00, 87.5MB/s]

Raw Dataset Shape: (541909, 8)
  InvoiceNo StockCode                         Description  Quantity  \
0    536365    85123A  WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                 WHITE METAL LANTERN         6   
2    536365    84406B      CREAM CUPID HEARTS COAT HANGER         8   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  


In [5]:
# A. Filter out guest checkouts (missing CustomerID)
df = df.dropna(subset=['CustomerID'])

# B. Drop cancelled orders and refunds (Quantity <= 0) and zero/negative prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# C. Cast CustomerID to clean integer
df['CustomerID'] = df['CustomerID'].astype(int)

# D. Compute total revenue for each transaction line
df['TotalSpend'] = df['Quantity'] * df['UnitPrice']

# E. Parse datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# F. Set baseline date: 1 day past the latest purchase in the log
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# G. Aggregate into customer-level RFM metrics
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalSpend': 'sum'                                      # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print("\n--- RFM Table Ready! ---")
print("Total Unique Customers:", rfm.shape[0])
print(rfm.head())


--- RFM Table Ready! ---
Total Unique Customers: 4338
   CustomerID  Recency  Frequency  Monetary
0       12346      326          1  77183.60
1       12347        2          7   4310.00
2       12348       75          4   1797.24
3       12349       19          1   1757.55
4       12350      310          1    334.40
